## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Data Import 

In [2]:
fatherhood = pd.read_csv("Fatherhood.csv")

participation = pd.read_csv("Participation.csv")

## Convert Client ID from float to string to allow merge

In [3]:
survey["Client ID"] = (
    survey["Client ID"]
    .astype("Int64")
    .astype("string")
)

participation["Client ID"] = (
    participation["Client ID"]
    .astype("Int64")
    .astype("string")
)

## Drop blank columns ##

In [4]:
survey = survey.dropna(axis=1, how='all')
participation = participation.dropna(axis=1, how='all')

## Replace empty and whitespace cells ##

Blank string values were standardized to NaN using regex-based replacement to ensure consistent missing value handling across the dataset. This step aligned all forms of empty responses (e.g., whitespace-only entries) with pandas’ native missing value representation, enabling accurate filtering and aggregation in downstream analysis and Tableau/Power BI visualizations.

Note: Null values were retained in the dataset to preserve data integrity, but excluded from most visualizations where they were not analytically relevant. Categorical fields contained minimal missingness, while missing numeric values were largely structural (i.e., dependent on survey design and respondent routing), rather than indicative of data quality issues. No imputation was performed, as missingness primarily reflected survey design and response structure rather than data quality issues.

In [5]:
survey = survey.replace(r'^\s*$', np.nan, regex=True)
participation = participation.replace(r'^\s*$', np.nan, regex=True)

## Keep only pertinent columns in 'participation' data frame

Most fields in 'participation' dataset are redundant/identical to those in fatherhood dataset. This command only keeps Client ID for merging and Primary Workshop Hours for additional analysis. 

In [35]:
keep_columns = ["Client ID", "Participation in Primary Workshop(s) Hours"]

participation = participation[keep_columns]

In [7]:
participation.head(1)

,Client ID,Participation in Primary Workshop(s) Hours
0,10845077,16


## Merge datasets on Client ID

In [8]:
fatherhood = survey.merge(
    participation,
    on="Client ID",
    how="left",
    validate="one_to_one"
)

In [9]:
fatherhood.head(0)

,Client ID,Office Location,Program,Client Population,Enrollment Date,Current Client Status,Client Status Change Date,Exit Survey Completion Date,ACS-Age,ACS-Ethnicity,...,EXIT-Child2_L1 Watched M,EXIT-Child2_L1 Sleep,EXIT-Child2_L1 Toys,EXIT-Child2_L1 Talk,EXIT-Child2_L1 Hug,EXIT-Child2_L1 Love,EXIT-Child2_L1 Sung,EXIT-Child2_L1 Read,EXIT-Child2_L1 Stories,Participation in Primary Workshop(s) Hours


In [12]:
fatherhood_clean = fatherhood.drop(
    columns=[
        "Program",
        "Client Population",
        "Client Status Change Date"
    ]
)

In [13]:
fatherhood_clean.head(0)

,Client ID,Office Location,Enrollment Date,Current Client Status,Exit Survey Completion Date,ACS-Age,ACS-Ethnicity,ACS-Race 1,ACS-Race 2,ACS-Race 3,...,EXIT-Child2_L1 Watched M,EXIT-Child2_L1 Sleep,EXIT-Child2_L1 Toys,EXIT-Child2_L1 Talk,EXIT-Child2_L1 Hug,EXIT-Child2_L1 Love,EXIT-Child2_L1 Sung,EXIT-Child2_L1 Read,EXIT-Child2_L1 Stories,Participation in Primary Workshop(s) Hours


## Replace Coded Survey Responses

In [14]:
fatherhood_clean["Current Client Status"] = fatherhood_clean["Current Client Status"].replace({
    1: "Applicant Pending Enrollment",
    4: "Duplicate Confirmed",
    5: "Completed/Graduated",
    6: "Drop Out",
    7: "Non-responsive",
    8: "Temporary Hold",
    9: "Moved out of Area",
    10: "Incarcerated",
    12: "Consent Revoked",
    17: "Deceased",
})

In [15]:
fatherhood_clean["ACS-Ethnicity"] = fatherhood_clean["ACS-Ethnicity"].replace({
    1: "Not Hispanic or Latino",
    2: "Hispanic or Latino",
})

In [17]:
fatherhood_clean["ACS-Language Primary"] = fatherhood_clean["ACS-Language Primary"].replace({
    1: "English",
    2: "Spanish",
    3: "English and Spanish equally",
    4: "Other",
})

In [18]:
fatherhood_clean["ACS-Living Situation"] = fatherhood_clean["ACS-Living Situation"].replace({
    1: "Own Home",
    2: "Rent",
    3: "Live with Parents or Relatives",
    4: "Live with Friends",
    5: "Shelter/Halfway House/Treatment Facility",
    6: "Unhoused",
    7: "Incarcerated",
    8: "Other",
})

In [19]:
fatherhood_clean["ACS-Living Area"] = fatherhood_clean["ACS-Living Area"].replace({
    1: "Large City",
    2: "Suburb of a Large City",
    3: "Small City",
    4: "Rural",
})

In [20]:
fatherhood_clean["ACS-Highest Degr"] = fatherhood_clean["ACS-Highest Degr"].replace({
    1: "None",
    2: "GED",
    3: "High School Diploma",
    4: "Vocational/Technical Cert",
    5: "Some College, no Degree",
    6: "Associate's Degree",
    7: "Bachelor's Degree",
    8: "Master's/Advanced Degree",
})

## Delete rows with confirmed duplicate status/revoked consent/pending enrollment 

(indicates no other response data)

In [22]:
counts = fatherhood_clean['Current Client Status'].value_counts()
counts

Current Client Status
Completed/Graduated             1139
Non-responsive                   252
Moved out of Area                 86
Drop Out                          49
Incarcerated                       9
Deceased                           5
Applicant Pending Enrollment       4
Consent Revoked                    2
Temporary Hold                     2
Duplicate Confirmed                1
Name: count, dtype: int64

In [29]:
exclude = ["Applicant Pending Enrollment", "Consent Revoked", "Duplicate Confirmed"]

fatherhood_clean = fatherhood_clean[fatherhood_clean["Current Client Status"].isin(exclude) == False]

In [32]:
counts2 = fatherhood_clean['Current Client Status'].value_counts()
counts2

Current Client Status
Completed/Graduated    1139
Non-responsive          252
Moved out of Area        86
Drop Out                 49
Incarcerated              9
Deceased                  5
Temporary Hold            2
nan                       1
Name: count, dtype: int64

In [33]:
fatherhood_clean.to_csv('Fatherhood Cleaned.csv', index=False)